In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from pyspark.sql.functions import col, count, isnan, when
from pyspark.sql.functions import col
from pyspark.sql.functions import to_date
import math
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score



In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AppleStockPricenew") \
    .getOrCreate()

df = spark.read.csv("hdfs:///user1/Apple1.csv", header=True, inferSchema=True)
df.show(5)


25/10/11 12:52:45 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+-------------------+------------------+------------------+------------------+------------------+---------+---------+------------+
|               Date|              Open|              High|               Low|             Close|   Volume|Dividends|Stock Splits|
+-------------------+------------------+------------------+------------------+------------------+---------+---------+------------+
|2021-01-04 05:00:00|130.10136410422703|130.18905617994074|123.51444475784658|126.09659576416016|143301900|      0.0|         0.0|
|2021-01-05 05:00:00|125.58986443353002|128.36689819774932|125.14163585794874|127.65557861328125| 97664900|      0.0|         0.0|
|2021-01-06 05:00:00|124.44986199518105|127.69460255545032|123.14416745080455|123.35853576660156|155088000|      0.0|         0.0|
|2021-01-07 05:00:00|125.07346559980046|128.25974454135618|124.58626762143851|127.56791687011719|109578200|      0.0|         0.0|
|2021-01-08 05:00:00| 129.0392208425577|129.23411188523488|126.89555318600054|128.6

In [3]:
df.printSchema()

root
 |-- Date: timestamp (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: integer (nullable = true)
 |-- Dividends: double (nullable = true)
 |-- Stock Splits: double (nullable = true)



In [4]:

# Separate numeric and non-numeric columns
numeric_cols = [c[0] for c in df.dtypes if c[1] in ('int', 'double', 'float')]
non_numeric_cols = [c[0] for c in df.dtypes if c[1] not in ('int', 'double', 'float')]

# Count nulls for numeric columns (check for NaN as well)
numeric_nulls = df.select([count(when(col(c).isNull() | isnan(c), c)).alias(c) for c in numeric_cols])

# Count nulls for non-numeric columns (only check for NULL)
non_numeric_nulls = df.select([count(when(col(c).isNull(), c)).alias(c) for c in non_numeric_cols])

# Show results
print("Numeric column nulls:")
numeric_nulls.show()
print("Non-numeric column nulls:")
non_numeric_nulls.show()


Numeric column nulls:


+----+----+---+-----+------+---------+------------+
|Open|High|Low|Close|Volume|Dividends|Stock Splits|
+----+----+---+-----+------+---------+------------+
|   0|   0|  0|    0|     0|        0|           0|
+----+----+---+-----+------+---------+------------+

Non-numeric column nulls:
+----+
|Date|
+----+
|   0|
+----+



In [5]:


# Select useful columns and drop/remove missing values (if any)
df_clean = df.select("Date", "Close", "Open", "High", "Low", "Volume").na.drop()

# Convert Date to proper format
df_clean = df_clean.withColumn("Date", to_date(col("Date")))
df_clean.schema["Date"].dataType


DateType()

In [6]:
# Sort by date
df_clean = df_clean.orderBy("Date")

In [7]:
#Converting to pandas dataframe
df_pd = df_clean.toPandas()
df_pd.shape



(1193, 6)